# MIDOG 2021 training set: download, convert, and package as a Kaggle dataset

Runs in a Kaggle notebook with **Internet on** (Settings > Internet). It downloads the 200 TIFF images from Zenodo **one at a time**,
converts each to a JPEG (quality 95, no chroma subsampling), deletes the original, and writes annotation tables.
Only the converted data (a few GB, not 26 GB) is kept in `/kaggle/working`.

Set `IDS` to `range(1, 11)` in the first cell for a quick test before the full run. The full run is resumable: finished images are skipped.

In [ ]:
import os, json, time, csv
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageDraw
Image.MAX_IMAGE_PIXELS = None

RECORD = "4643381"   # MIDOG 2021 training set (Zenodo)
BASE = os.environ.get("MIDOG_BASE", f"https://zenodo.org/records/{RECORD}/files")
OUT = Path(os.environ.get("MIDOG_OUT", "/kaggle/working/midog2021"))
RAW = Path(os.environ.get("MIDOG_RAW", "/kaggle/temp/midog_raw"))
IDS = [int(x) for x in os.environ["MIDOG_IDS"].split(",")] if "MIDOG_IDS" in os.environ else list(range(1, 201))
JPEG_QUALITY = 95

# scanner blocks as listed on the Zenodo page
SCANNER_RANGES = [(1, 50, "hamamatsu_xr"), (51, 100, "hamamatsu_s360"), (101, 150, "aperio_cs2"), (151, 200, "leica_gt450")]
def scanner_of(i):
    for lo, hi, name in SCANNER_RANGES:
        if lo <= i <= hi:
            return name
    raise ValueError(i)

for d in (OUT / "images", OUT / "check"):
    d.mkdir(parents=True, exist_ok=True)
try:
    RAW.mkdir(parents=True, exist_ok=True)
except OSError:
    RAW = Path("/tmp/midog_raw"); RAW.mkdir(parents=True, exist_ok=True)

def download(name, dest, tries=5):
    url = f"{BASE}/{name}?download=1"
    for attempt in range(1, tries + 1):
        try:
            tmp = dest.with_name(dest.name + ".part")
            with requests.get(url, stream=True, timeout=(30, 120)) as r:
                r.raise_for_status()
                want = int(r.headers.get("Content-Length", 0))
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            got = tmp.stat().st_size
            if want and got != want:
                raise IOError(f"incomplete download: {got} of {want} bytes")
            tmp.rename(dest)
            return
        except Exception as e:
            print(f"  {name}: attempt {attempt}/{tries} failed: {e}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"could not download {name}")
print("images to process:", len(IDS), "| output:", OUT)

## 1. Annotations: download and inspect
Check the printed category names and the bounding-box format guess before the long download.

In [ ]:
ann_file = RAW / "MIDOG.json"
if not ann_file.exists():
    download("MIDOG.json", ann_file)
coco = json.loads(ann_file.read_text())
print("top-level keys:", list(coco.keys()))
cats = {c["id"]: c["name"] for c in coco["categories"]}
print("categories:", cats)

mitotic_ids = [k for k, v in cats.items() if "mitotic" in v.lower() and not v.lower().startswith(("non", "not", "hard"))]
if len(mitotic_ids) != 1:
    print("WARNING: could not identify the mitotic category by name; assuming id 1")
    mitotic_ids = [1]
MITOTIC = mitotic_ids[0]
print("mitotic category id:", MITOTIC, "->", cats.get(MITOTIC))

anns = coco["annotations"]
print("annotations:", len(anns), "| per category:", pd.Series([a["category_id"] for a in anns]).value_counts().to_dict())
print("example annotation:", anns[0])
print("example image record:", coco["images"][0])

# bbox format: the true width is the one that varies least across boxes
bb = np.array([a["bbox"] for a in anns], dtype=float)
cv = lambda v: v.std() / max(abs(v.mean()), 1e-9)
BBOX_FMT = "xyxy" if cv(bb[:, 2] - bb[:, 0]) < cv(bb[:, 2]) else "xywh"
w = (bb[:, 2] - bb[:, 0]) if BBOX_FMT == "xyxy" else bb[:, 2]
h = (bb[:, 3] - bb[:, 1]) if BBOX_FMT == "xyxy" else bb[:, 3]
print(f"bbox format guess: {BBOX_FMT}; width median {np.median(w):.0f} px (min {w.min():.0f}, max {w.max():.0f}), height median {np.median(h):.0f} px")

def to_xyxy(b):
    x, y, c, d = [float(v) for v in b]
    return (x, y, c, d) if BBOX_FMT == "xyxy" else (x, y, x + c, y + d)
by_stem = {Path(im["file_name"]).stem: im for im in coco["images"]}

## 2. Download, convert, delete the original
Resumable. Expect roughly 30 to 90 minutes in total, depending on Zenodo speed.

In [ ]:
t0 = time.time()
done = 0
for n, i in enumerate(IDS, 1):
    stem = f"{i:03d}"
    jpg = OUT / "images" / f"{stem}.jpg"
    if jpg.exists():
        continue
    raw = RAW / f"{stem}.tiff"
    download(f"{stem}.tiff", raw)
    img = Image.open(raw).convert("RGB")
    meta = by_stem[stem]
    if "width" in meta and (img.width, img.height) != (meta["width"], meta["height"]):
        print(f"  WARNING {stem}: image is {img.width}x{img.height}, JSON says {meta['width']}x{meta['height']}")
    part = jpg.with_name(stem + ".part")
    img.save(part, format="JPEG", quality=JPEG_QUALITY, subsampling=0)
    part.rename(jpg)
    size_mb = jpg.stat().st_size / 1e6
    img.close(); raw.unlink()
    done += 1
    print(f"[{n}/{len(IDS)}] {stem} {scanner_of(i)} {img.width}x{img.height} -> {size_mb:.1f} MB JPEG | elapsed {(time.time()-t0)/60:.1f} min", flush=True)
print("converted this session:", done)

## 3. Write annotation tables and README

In [ ]:
img_rows, box_rows = [], []
for i in IDS:
    stem = f"{i:03d}"
    meta = by_stem[stem]
    with Image.open(OUT / "images" / f"{stem}.jpg") as im:
        W, H = im.size
    mine = [a for a in anns if a["image_id"] == meta["id"]]
    n_m = sum(a["category_id"] == MITOTIC for a in mine)
    img_rows.append(dict(image=f"{stem}.jpg", scanner=scanner_of(i), width=W, height=H, n_mitotic=n_m, n_hard_negative=len(mine) - n_m))
    for a in mine:
        x1, y1, x2, y2 = to_xyxy(a["bbox"])
        box_rows.append(dict(image=f"{stem}.jpg", scanner=scanner_of(i), x1=x1, y1=y1, x2=x2, y2=y2,
                             label="mitotic" if a["category_id"] == MITOTIC else "hard_negative"))
images_df, boxes_df = pd.DataFrame(img_rows), pd.DataFrame(box_rows)
images_df.to_csv(OUT / "images.csv", index=False)
boxes_df.to_csv(OUT / "annotations.csv", index=False)

(OUT / "README.txt").write_text(f"""MIDOG 2021 training set, converted for experiments.
Source: https://zenodo.org/records/{RECORD} (check the license and citation requirements on that page; keep this dataset private unless the license allows redistribution).
Images: original TIFFs converted to JPEG (quality {JPEG_QUALITY}, 4:4:4). Boxes are in original pixel coordinates (x2, y2 exclusive-style as given by the source).
annotations.csv: image, scanner, x1, y1, x2, y2, label (mitotic | hard_negative). images.csv: per-image size and counts.
Scanner blocks: 001-050 hamamatsu_xr, 051-100 hamamatsu_s360, 101-150 aperio_cs2, 151-200 leica_gt450.
Each image is a different case (200 cases), so scanners do not share tissue.
""")
print(images_df.groupby("scanner").agg(images=("image", "count"), mitotic=("n_mitotic", "sum"), hard_neg=("n_hard_negative", "sum"),
                                        width_med=("width", "median"), height_med=("height", "median")))
tot = sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e9
print(f"total output size: {tot:.2f} GB (Kaggle working folder limit is about 20 GB)")

## 4. Check that the boxes sit on the cells
Green = mitotic figure, red = hard negative. Look at the crops before using the data.

In [ ]:
rng = np.random.default_rng(0)
pick = rng.choice(len(images_df), size=min(4, len(images_df)), replace=False)
tiles = []
for k in pick:
    name = images_df.image[k]
    sub = boxes_df[(boxes_df.image == name) & (boxes_df.label == "mitotic")]
    if sub.empty:
        continue
    r = sub.iloc[0]
    cx, cy = int((r.x1 + r.x2) / 2), int((r.y1 + r.y2) / 2)
    with Image.open(OUT / "images" / name) as im:
        x0, y0 = max(cx - 128, 0), max(cy - 128, 0)
        crop = im.crop((x0, y0, x0 + 256, y0 + 256)).convert("RGB")
    d = ImageDraw.Draw(crop)
    for _, b in boxes_df[boxes_df.image == name].iterrows():
        d.rectangle([b.x1 - x0, b.y1 - y0, b.x2 - x0, b.y2 - y0], outline=(0, 255, 0) if b.label == "mitotic" else (255, 0, 0), width=2)
    crop = crop.resize((384, 384)); crop.save(OUT / "check" / f"check_{name}")
    tiles.append(crop)
if tiles:
    sheet = Image.new("RGB", (384 * len(tiles), 384))
    for j, t in enumerate(tiles):
        sheet.paste(t, (384 * j, 0))
    sheet.save(OUT / "check" / "overview.jpg")
    try:
        from IPython.display import display; display(sheet)
    except Exception:
        print("saved", OUT / "check" / "overview.jpg")

## 5. Turn the output into a Kaggle dataset

**Easiest (no API key):**
1. Click **Save Version > Save & Run All (Commit)** and wait for it to finish.
2. Open the finished version, go to the **Output** tab, and click **New Dataset**. Name it, for example, `midog2021-jpeg`, and keep it **Private**.
3. Delete `check/` from the dataset if you do not want the preview images in it.

In later notebooks add the dataset through **Add Input**; it appears under `/kaggle/input/midog2021-jpeg/`.

**Optional, from inside the notebook (needs a Kaggle API token stored as the secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`):** run the next cell.

In [ ]:
# OPTIONAL: create the dataset directly through the Kaggle API. Uncomment to use.
# from kaggle_secrets import UserSecretsClient
# s = UserSecretsClient(); os.environ["KAGGLE_USERNAME"] = s.get_secret("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = s.get_secret("KAGGLE_KEY")
# meta = {"title": "MIDOG 2021 training set (JPEG)", "id": f"{os.environ['KAGGLE_USERNAME']}/midog2021-jpeg", "licenses": [{"name": "other"}]}
# (OUT / "dataset-metadata.json").write_text(json.dumps(meta))
# !kaggle datasets create -p {OUT} --dir-mode zip